In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Pass Transpiler AI
Pass Transpiler berbasis AI adalah pass yang berfungsi sebagai pengganti langsung dari pass Qiskit "tradisional" untuk beberapa tugas transpilasi. Pass ini sering menghasilkan hasil yang lebih baik dibandingkan algoritma heuristik yang ada (seperti depth dan jumlah CNOT yang lebih rendah), tetapi juga jauh lebih cepat daripada algoritma optimasi seperti solver Boolean satisfiability. Pass Transpiler AI dapat dijalankan di lingkungan lokal kamu atau di cloud menggunakan Qiskit Transpiler Service jika kamu termasuk dalam IBM Quantum&reg; Premium Plan, Flex Plan, atau On-Prem (via IBM Quantum Platform API) Plan.

> **Note:** Pass Transpiler berbasis AI masih dalam status rilis beta dan dapat berubah sewaktu-waktu.
>     Jika kamu punya masukan atau ingin menghubungi tim developer, silakan gunakan [channel Qiskit Slack Workspace ini](https://qiskit.slack.com/archives/C06KF8YHUAU).

Pass yang tersedia saat ini adalah sebagai berikut:

**Pass Routing**
 - `AIRouting`: Pemilihan layout dan routing Circuit

**Pass sintesis Circuit**
 - `AICliffordSynthesis`: Sintesis Circuit Clifford
 - `AILinearFunctionSynthesis`: Sintesis Circuit fungsi linear
 - `AIPermutationSynthesis`: Sintesis Circuit permutasi
 - `AIPauliNetworkSynthesis`: Sintesis Circuit Pauli Network

Untuk menggunakan pass Transpiler AI, pertama [instal paket `qiskit-ibm-transpiler`](/guides/qiskit-transpiler-service#install-transpiler-service). Kunjungi [dokumentasi API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) untuk mendapatkan informasi lebih lanjut tentang berbagai opsi yang tersedia.

## Jalankan pass Transpiler AI secara lokal atau di cloud
Jika kamu ingin menggunakan pass Transpiler berbasis AI di lingkungan lokal secara gratis, instal `qiskit-ibm-transpiler` dengan beberapa dependensi tambahan sebagai berikut:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService

backend = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Tanpa dependensi tambahan ini, pass Transpiler berbasis AI akan berjalan di cloud melalui Qiskit Transpiler Service (hanya tersedia untuk pengguna IBM Quantum Premium Plan, Flex Plan, atau On-Prem (via IBM Quantum Platform API) Plan). Setelah menginstal dependensi tambahan, mode default untuk menjalankan pass Transpiler berbasis AI adalah menggunakan mesin lokal kamu.

## Pass routing AI
Pass `AIRouting` berfungsi sebagai stage layout sekaligus stage routing. Pass ini dapat digunakan di dalam `PassManager` sebagai berikut:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit.circuit.library import efficient_su2

ibm_torino = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_torino,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Linear Function blocks
        CollectPauliNetworks(),  # Collect Pauli Networks blocks
        AIPauliNetworkSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Pauli Network blocks.
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Di sini, `backend` menentukan coupling map mana yang akan digunakan untuk routing, `optimization_level` (1, 2, atau 3) menentukan seberapa besar usaha komputasi yang digunakan dalam prosesnya (nilai lebih tinggi biasanya menghasilkan hasil yang lebih baik tetapi membutuhkan waktu lebih lama), dan `layout_mode` menentukan cara menangani pemilihan layout.
`layout_mode` mencakup opsi berikut:

- `keep`: Mode ini menghormati layout yang ditetapkan oleh pass Transpiler sebelumnya (atau menggunakan trivial layout jika belum diatur). Biasanya hanya digunakan ketika Circuit harus dijalankan pada Qubit tertentu di perangkat. Mode ini sering menghasilkan hasil yang lebih buruk karena ruang optimasinya lebih terbatas.
- `improve`: Mode ini menggunakan layout yang ditetapkan oleh pass Transpiler sebelumnya sebagai titik awal. Berguna ketika kamu memiliki tebakan awal yang baik untuk layout; misalnya, untuk Circuit yang dibangun dengan cara yang kurang lebih mengikuti coupling map perangkat. Mode ini juga berguna jika kamu ingin mencoba pass layout tertentu lainnya dikombinasikan dengan pass `AIRouting`.
- `optimize`: Ini adalah mode default. Paling cocok untuk Circuit umum di mana kamu mungkin tidak memiliki tebakan layout yang baik. Mode ini mengabaikan pemilihan layout sebelumnya.
- `local_mode`: Flag ini menentukan di mana pass `AIRouting` dijalankan. Jika `False`, `AIRouting` berjalan secara remote melalui Qiskit Transpiler Service. Jika `True`, paket akan mencoba menjalankan pass di lingkungan lokal kamu dengan fallback ke mode cloud jika dependensi yang diperlukan tidak ditemukan.

## Pass sintesis Circuit AI
Pass sintesis Circuit AI memungkinkan kamu untuk mengoptimalkan potongan-potongan dari berbagai jenis Circuit ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network) dengan cara men-sintesis ulang. Cara umum untuk menggunakan pass sintesis adalah sebagai berikut:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_torino")
torino_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=torino_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

Proses sintesis menghormati coupling map perangkat: sintesis dapat dijalankan dengan aman setelah pass routing lainnya tanpa mengganggu Circuit, sehingga Circuit keseluruhan tetap mengikuti batasan perangkat. Secara default, sintesis hanya akan mengganti sub-Circuit asli jika sub-Circuit hasil sintesis lebih baik dari yang asli (saat ini hanya memeriksa jumlah CNOT), tetapi ini dapat dipaksa untuk selalu mengganti Circuit dengan menetapkan `replace_only_if_better=False`.

Pass sintesis berikut tersedia dari `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Sintesis untuk Circuit [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford) (blok Gate `H`, `S`, dan `CX`). Saat ini mendukung hingga sembilan blok Qubit.
- *AILinearFunctionSynthesis*: Sintesis untuk Circuit [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) (blok Gate `CX` dan `SWAP`). Saat ini mendukung hingga sembilan blok Qubit.
- *AIPermutationSynthesis*: Sintesis untuk Circuit [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation) (blok Gate `SWAP`). Saat ini tersedia untuk blok 65, 33, dan 27 Qubit.
- *AIPauliNetworkSynthesis*: Sintesis untuk Circuit Pauli Network (blok Gate `H`, `S`, `SX`, `CX`, `RX`, `RY` dan `RZ`). Saat ini mendukung hingga enam blok Qubit.

Kami berencana untuk secara bertahap meningkatkan ukuran blok yang didukung.

Semua pass menggunakan thread pool untuk mengirim beberapa permintaan secara paralel. Secara default, jumlah maksimum thread adalah jumlah core ditambah empat (nilai default untuk objek Python `ThreadPoolExecutor`). Namun, kamu dapat menetapkan nilai sendiri dengan argumen `max_threads` saat instantiasi pass. Sebagai contoh, baris berikut meng-instantiasi pass `AILinearFunctionSynthesis`, yang memungkinkannya menggunakan maksimum 20 thread.